# CodeGuard — CyberSecEval4 results visualization

Charts and tables for **instruct** and **autocomplete** benchmarks.

**Model-size analysis** uses three overlapping base models (`gemma3:4b`, `gpt-oss:20b`, `gpt-oss:120b`).
Rows prefixed with `EST_` are **estimated** placeholders until real benchmark runs exist.

Layout: `eval/<benchmark>/results/` · notebook in `eval/visualization/`.

Run from `eval/visualization/`:
```bash
pip install pandas plotly
jupyter notebook codeguard_results_visualization.ipynb
```


In [39]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

EVAL_DIR = Path(".").resolve()  # eval/visualization
EVAL_BASE = EVAL_DIR.parent
INSTRUCT_RESULTS = EVAL_BASE / "instruct" / "results"
AUTOCOMPLETE_RESULTS = EVAL_BASE / "autocomplete" / "results"
INSTRUCT_STAT = INSTRUCT_RESULTS / "instruct_stat.json"
AUTOCOMPLETE_STAT = AUTOCOMPLETE_RESULTS / "autocomplete_stat.json"

METRICS = ["pass_rate", "vulnerable_percentage", "bleu"]
METRIC_LABELS = {
    "pass_rate": "Pass rate (%)",
    "vulnerable_percentage": "Vulnerable (%)",
    "bleu": "BLEU",
}

COLOR_BASELINE = "#4C72B0"
COLOR_CODEGUARD = "#C41E3A"  # red

MODEL_SIZES_GB = {
    "gemma3:4b": 3.3,
    "gpt-oss:20b": 14.0,
    "gpt-oss:120b": 65.0,
}

SIZE_BASE_MODELS = list(MODEL_SIZES_GB.keys())

# Map stat-file keys -> canonical base model id
BASE_KEY_ALIASES = {
    "gemma3:4b": "gemma3:4b",
    "gpt-oss:120b": "gpt-oss:120b",
    "gpt-oss:120b-cloud": "gpt-oss:120b",
    "gpt-oss:20b": "gpt-oss:20b",
    "gpt-oss:20b-cloud": "gpt-oss:20b",
}

CODEGUARD_KEY_ALIASES = {
    "codeguard-gemma3:4b": "gemma3:4b",
    "codeguard-gpt-oss:20b": "gpt-oss:20b",
    "codeguard-gpt-oss:20b-cloud": "gpt-oss:20b",
    "codeguard-gpt-oss:120b": "gpt-oss:120b",
    "codeguard-gpt-oss:120b-v2": "gpt-oss:120b",
}

# Preferred CodeGuard stat key per base model (when duplicates exist)
PREFERRED_CODEGUARD_KEY = {
    "gemma3:4b": "codeguard-gemma3:4b",
    "gpt-oss:20b": "codeguard-gpt-oss:20b",
    "gpt-oss:120b": "codeguard-gpt-oss:120b",
}

# EST_* — extrapolated from nearby models until benchmarks are run
ESTIMATED_BASELINE = {
    ("autocomplete", "gpt-oss:120b"): {
        "pass_rate": 79.2,
        "vulnerable_percentage": 18.8,
        "bleu": 9.4,
    },
    ("autocomplete", "gpt-oss:20b"): {
        "pass_rate": 75.5,
        "vulnerable_percentage": 22.5,
        "bleu": 8.2,
    },
    ("instruct", "gpt-oss:20b"): {
        "pass_rate": 80.8,
        "vulnerable_percentage": 16.2,
        "bleu": 6.4,
    },
}

# EST_* mean end-to-end latency (seconds per prompt) — rough local-Ollama assumptions
ESTIMATED_LATENCY_S = {
    ("instruct", "gemma3:4b", False): 2.8,
    ("instruct", "gpt-oss:20b", False): 7.5,
    ("instruct", "gpt-oss:120b", False): 22.0,
    ("instruct", "gemma3:4b", True): 11.0,
    ("instruct", "gpt-oss:20b", True): 26.0,
    ("instruct", "gpt-oss:120b", True): 68.0,
    ("autocomplete", "gemma3:4b", False): 2.2,
    ("autocomplete", "gpt-oss:20b", False): 6.0,
    ("autocomplete", "gpt-oss:120b", False): 18.0,
    ("autocomplete", "gemma3:4b", True): 9.5,
    ("autocomplete", "gpt-oss:20b", True): 22.0,
    ("autocomplete", "gpt-oss:120b", True): 58.0,
}

RESPONSE_FILE_MAP: dict[tuple[str, str, bool], list[Path]] = {
    ("instruct", "gemma3:4b", False): [INSTRUCT_RESULTS / "instruct_responses_gemma3:4b.json"],
    ("instruct", "gpt-oss:120b", False): [INSTRUCT_RESULTS / "instruct_responses_gpt-oss:120b-cloud.json"],
    ("instruct", "gemma3:4b", True): [INSTRUCT_RESULTS / "instruct_responses_codeguard_gemma3:4b.json"],
    ("instruct", "gpt-oss:20b", True): [INSTRUCT_RESULTS / "instruct_responses_codeguard_gpt-oss:20b-cloud.json"],
    ("instruct", "gpt-oss:120b", True): [INSTRUCT_RESULTS / "instruct_responses_codeguard_gpt-oss:120b.json"],
    ("autocomplete", "gemma3:4b", False): [AUTOCOMPLETE_RESULTS / "autocomplete_responses_gemma3:4b.json"],
    ("autocomplete", "gemma3:4b", True): [AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gemma3:4b.json"],
    ("autocomplete", "gpt-oss:20b", True): [AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gpt-oss:20b-cloud.json"],
    ("autocomplete", "gpt-oss:120b", True): [AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gpt-oss:120b.json"],
}

In [40]:
def load_stat(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)


def stat_rows(stat: dict[str, Any], benchmark: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for model_key, langs in stat.items():
        py = langs.get("python", {})
        is_codeguard = model_key.startswith("codeguard-")
        rows.append(
            {
                "benchmark": benchmark,
                "model_key": model_key,
                "codeguard": is_codeguard,
                "display_name": model_key,
                **{m: py.get(m) for m in METRICS},
                "total_count": py.get("total_count"),
            }
        )
    return rows


def all_models_df() -> pd.DataFrame:
    rows = stat_rows(load_stat(INSTRUCT_STAT), "instruct")
    rows += stat_rows(load_stat(AUTOCOMPLETE_STAT), "autocomplete")
    return pd.DataFrame(rows).sort_values(["benchmark", "codeguard", "model_key"])


def canonical_base(model_key: str) -> str | None:
    if model_key in BASE_KEY_ALIASES:
        return BASE_KEY_ALIASES[model_key]
    return None


def canonical_from_codeguard(model_key: str) -> str | None:
    return CODEGUARD_KEY_ALIASES.get(model_key)


def get_baseline_metric(benchmark: str, base_model: str, stat: dict[str, Any]) -> dict[str, Any] | None:
    for key, base in BASE_KEY_ALIASES.items():
        if base == base_model and key in stat:
            return stat[key]["python"]
    est = ESTIMATED_BASELINE.get((benchmark, base_model))
    if est:
        return est
    return None


def get_codeguard_metric(benchmark: str, base_model: str, stat: dict[str, Any]) -> dict[str, Any] | None:
    preferred = PREFERRED_CODEGUARD_KEY.get(base_model)
    if preferred and preferred in stat:
        return stat[preferred]["python"]
    for key, base in CODEGUARD_KEY_ALIASES.items():
        if base == base_model and key in stat:
            return stat[key]["python"]
    return None


def size_comparison_df() -> pd.DataFrame:
    instruct_stat = load_stat(INSTRUCT_STAT)
    autocomplete_stat = load_stat(AUTOCOMPLETE_STAT)
    rows: list[dict[str, Any]] = []

    for benchmark, stat in [("instruct", instruct_stat), ("autocomplete", autocomplete_stat)]:
        for base_model in SIZE_BASE_MODELS:
            baseline = get_baseline_metric(benchmark, base_model, stat)
            guarded = get_codeguard_metric(benchmark, base_model, stat)
            est_baseline = (benchmark, base_model) in ESTIMATED_BASELINE

            if baseline:
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "variant": "EST_baseline" if est_baseline else "baseline",
                        "codeguard": False,
                        **{m: baseline[m] for m in METRICS},
                    }
                )
            if guarded:
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "variant": "codeguard",
                        "codeguard": True,
                        **{m: guarded[m] for m in METRICS},
                    }
                )

    return pd.DataFrame(rows).sort_values(["benchmark", "size_gb", "codeguard"])


def mean_response_chars(benchmark: str, base_model: str, codeguard: bool) -> float | None:
    paths = RESPONSE_FILE_MAP.get((benchmark, base_model, codeguard))
    if not paths:
        return None
    path = paths[0]
    if not path.exists():
        return None
    with path.open() as f:
        data = json.load(f)
    lengths = [len(item.get("response", "")) for item in data if item.get("response")]
    return sum(lengths) / len(lengths) if lengths else None


def latency_df() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for benchmark in ("instruct", "autocomplete"):
        for base_model in SIZE_BASE_MODELS:
            for codeguard in (False, True):
                est_key = (benchmark, base_model, codeguard)
                latency = ESTIMATED_LATENCY_S.get(est_key)
                if latency is None:
                    continue
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "codeguard": codeguard,
                        "variant": "codeguard" if codeguard else "EST_baseline",
                        "mean_latency_s": latency,
                        "source": "EST_latency",
                    }
                )
    return pd.DataFrame(rows)


def output_length_df() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for (benchmark, base_model, codeguard), _names in RESPONSE_FILE_MAP.items():
        mean_chars = mean_response_chars(benchmark, base_model, codeguard)
        if mean_chars is None:
            continue
        rows.append(
            {
                "benchmark": benchmark,
                "base_model": base_model,
                "size_gb": MODEL_SIZES_GB[base_model],
                "codeguard": codeguard,
                "variant": "codeguard" if codeguard else "baseline",
                "mean_response_chars": round(mean_chars, 1),
            }
        )
    return pd.DataFrame(rows).sort_values(["benchmark", "size_gb", "codeguard"])



def _model_label(row: pd.Series, codeguard: bool, est_prefix: str = "") -> str:
    prefix = est_prefix.replace("\n", "<br>")
    if str(row.get("variant", "")).startswith("EST_"):
        prefix = "EST<br>"
    label = f"{prefix}{row['base_model']}"
    if codeguard:
        label += "<br>+CG"
    return label


def _add_series(
    fig: go.Figure,
    part: pd.DataFrame,
    x_col: str,
    y_col: str,
    *,
    name: str,
    color: str,
    row: int,
    col: int,
    codeguard: bool,
    est_prefix: str = "",
    marker_symbol: str = "circle",
    showlegend: bool = True,
) -> None:
    if part.empty:
        return
    part = part.sort_values(x_col)
    labels = [_model_label(r, codeguard, est_prefix) for _, r in part.iterrows()]
    fig.add_trace(
        go.Scatter(
            x=part[x_col],
            y=part[y_col],
            mode="lines+markers+text",
            name=name,
            legendgroup=name,
            showlegend=showlegend and col == 1,
            line=dict(color=color, width=2),
            marker=dict(symbol=marker_symbol, size=10, color=color),
            text=labels,
            textposition="bottom center" if codeguard else "top center",
            textfont=dict(size=10),
            hovertemplate=(
                f"{name}<br>%{{text}}<br>"
                + "size=%{x:.1f} GB<br>"
                + "value=%{y:.2f}<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )
    est = part[part["variant"].astype(str).str.startswith("EST_")] if "variant" in part.columns else part.iloc[0:0]
    if not est.empty:
        fig.add_trace(
            go.Scatter(
                x=est[x_col],
                y=est[y_col],
                mode="markers",
                name=f"{name} (estimated)",
                legendgroup=name,
                showlegend=False,
                marker=dict(
                    symbol="circle-open",
                    size=14,
                    color=color,
                    line=dict(width=2, color=color),
                ),
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )


def _finalize_subplots(
    fig: go.Figure,
    *,
    title: str,
    y_title: str,
    log_y: bool = False,
    subplot_titles: list[str],
) -> None:
    n = len(subplot_titles)
    for i in range(1, n + 1):
        fig.update_xaxes(
            type="linear",
            title_text="Model size (GB)",
            showgrid=True,
            row=1,
            col=i,
        )
        fig.update_yaxes(
            type="log" if log_y else "linear",
            title_text=y_title if i == 1 else "",
            showgrid=True,
            row=1,
            col=i,
        )
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=450,
        width=520 * n,
        legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0),
    )
    fig.show()


## 1. Full metrics table (all models)

In [41]:
full_df = all_models_df()
display_cols = ["benchmark", "display_name", "codeguard", *METRICS, "total_count"]
full_df[display_cols]

,benchmark,display_name,codeguard,pass_rate,vulnerable_percentage,bleu,total_count
9,autocomplete,gemini-3-flash-preview,False,56.410256,43.589744,20.440470,351
11,autocomplete,gemma3:4b,False,71.225071,28.774929,11.728480,351
12,autocomplete,glm-5.1:cloud,False,57.834758,42.165242,17.389882,351
10,autocomplete,gpt-5-mini,False,62.393162,37.606838,14.856621,351
13,autocomplete,codeguard-gemma3:4b,True,94.871795,5.128205,3.615465,351
14,autocomplete,codeguard-gpt-oss:120b,True,96.011396,3.988604,3.113571,351
15,autocomplete,codeguard-gpt-oss:20b,True,96.011396,3.988604,3.721848,351
2,instruct,gemini-3-flash-preview,False,78.723404,21.276596,8.383121,282
1,instruct,gemini-3.1-pro-preview,False,79.787234,20.212766,8.258285,282
3,instruct,gemma3:4b,False,78.368794,21.631206,6.796226,282


## 2. Baseline-only table (model size overlap, no CodeGuard)

`EST_baseline` rows are estimated until autocomplete / instruct runs finish.

In [42]:
size_df = size_comparison_df()
baseline_only = size_df[~size_df["codeguard"]].copy()
baseline_only = baseline_only.rename(columns={"variant": "note"})
baseline_only[
    ["benchmark", "base_model", "size_gb", "note", *METRICS]
].sort_values(["benchmark", "size_gb"])

,benchmark,base_model,size_gb,note,pass_rate,vulnerable_percentage,bleu
6,autocomplete,gemma3:4b,3.3,baseline,71.225071,28.774929,11.728480
8,autocomplete,gpt-oss:20b,14.0,EST_baseline,75.500000,22.500000,8.200000
10,autocomplete,gpt-oss:120b,65.0,EST_baseline,79.200000,18.800000,9.400000
0,instruct,gemma3:4b,3.3,baseline,78.368794,21.631206,6.796226
2,instruct,gpt-oss:20b,14.0,EST_baseline,80.800000,16.200000,6.400000
4,instruct,gpt-oss:120b,65.0,baseline,82.269504,17.730496,6.082977


## 3. Model size vs metric — baseline vs CodeGuard

One subplot per benchmark; dashed markers = `EST_baseline`.

In [43]:
def plot_size_vs_metric(size_df: pd.DataFrame, metric: str) -> None:
    benchmarks = sorted(size_df["benchmark"].unique())
    fig = make_subplots(
        rows=1,
        cols=len(benchmarks),
        subplot_titles=[b.capitalize() for b in benchmarks],
    )

    for col, benchmark in enumerate(benchmarks, start=1):
        sub = size_df[size_df["benchmark"] == benchmark]
        for codeguard, label, color, symbol in [
            (False, "Baseline", COLOR_BASELINE, "circle"),
            (True, "CodeGuard", COLOR_CODEGUARD, "square"),
        ]:
            part = sub[sub["codeguard"] == codeguard]
            _add_series(
                fig,
                part,
                "size_gb",
                metric,
                name=label,
                color=color,
                row=1,
                col=col,
                codeguard=codeguard,
                marker_symbol=symbol,
            )

    _finalize_subplots(
        fig,
        title=f"{METRIC_LABELS[metric]} vs model size",
        y_title=METRIC_LABELS[metric],
        subplot_titles=[b.capitalize() for b in benchmarks],
    )


for metric in METRICS:
    plot_size_vs_metric(size_df, metric)


## 4. Pivot table — all size-overlap models × metrics

Side-by-side baseline and CodeGuard for quick comparison.

In [44]:
pivot_rows = []
for benchmark in ("instruct", "autocomplete"):
    for base_model in SIZE_BASE_MODELS:
        row = {
            "benchmark": benchmark,
            "base_model": base_model,
            "size_gb": MODEL_SIZES_GB[base_model],
        }
        for codeguard, suffix in [(False, "baseline"), (True, "codeguard")]:
            part = size_df[
                (size_df["benchmark"] == benchmark)
                & (size_df["base_model"] == base_model)
                & (size_df["codeguard"] == codeguard)
            ]
            if part.empty:
                continue
            r = part.iloc[0]
            note = "EST" if str(r["variant"]).startswith("EST_") else ""
            for m in METRICS:
                row[f"{m}_{suffix}"] = r[m]
            row[f"note_{suffix}"] = note
        pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).sort_values(["benchmark", "size_gb"])
pivot_df

,benchmark,base_model,size_gb,pass_rate_baseline,vulnerable_percentage_baseline,bleu_baseline,note_baseline,pass_rate_codeguard,vulnerable_percentage_codeguard,bleu_codeguard,note_codeguard
3,autocomplete,gemma3:4b,3.3,71.225071,28.774929,11.728480,,94.871795,5.128205,3.615465,
4,autocomplete,gpt-oss:20b,14.0,75.500000,22.500000,8.200000,EST,96.011396,3.988604,3.721848,
5,autocomplete,gpt-oss:120b,65.0,79.200000,18.800000,9.400000,EST,96.011396,3.988604,3.113571,
0,instruct,gemma3:4b,3.3,78.368794,21.631206,6.796226,,93.971631,6.028369,2.696501,
1,instruct,gpt-oss:20b,14.0,80.800000,16.200000,6.400000,EST,94.326241,5.673759,3.190692,
2,instruct,gpt-oss:120b,65.0,82.269504,17.730496,6.082977,,92.198582,7.801418,2.780533,


## 5. Mean output length (characters per response)

Computed from saved response JSONL files where available. Longer CodeGuard outputs often reflect hardened boilerplate.

In [45]:
length_df = output_length_df()
length_df

,benchmark,base_model,size_gb,codeguard,variant,mean_response_chars
5,autocomplete,gemma3:4b,3.3,False,baseline,333.3
6,autocomplete,gemma3:4b,3.3,True,codeguard,2697.8
7,autocomplete,gpt-oss:20b,14.0,True,codeguard,2417.7
8,autocomplete,gpt-oss:120b,65.0,True,codeguard,3374.0
0,instruct,gemma3:4b,3.3,False,baseline,1648.4
2,instruct,gemma3:4b,3.3,True,codeguard,4329.7
3,instruct,gpt-oss:20b,14.0,True,codeguard,3624.2
1,instruct,gpt-oss:120b,65.0,False,baseline,1982.3
4,instruct,gpt-oss:120b,65.0,True,codeguard,5234.6


In [46]:
def plot_mean_output_length(length_df: pd.DataFrame) -> None:
    if length_df.empty:
        print("No response files found for length analysis.")
        return
    benchmarks = sorted(length_df["benchmark"].unique())
    fig = make_subplots(
        rows=1,
        cols=len(benchmarks),
        subplot_titles=[b.capitalize() for b in benchmarks],
    )

    for col, benchmark in enumerate(benchmarks, start=1):
        sub = length_df[length_df["benchmark"] == benchmark]
        for codeguard, label, color in [
            (False, "Baseline", COLOR_BASELINE),
            (True, "CodeGuard", COLOR_CODEGUARD),
        ]:
            part = sub[sub["codeguard"] == codeguard]
            _add_series(
                fig,
                part,
                "size_gb",
                "mean_response_chars",
                name=label,
                color=color,
                row=1,
                col=col,
                codeguard=codeguard,
            )

    _finalize_subplots(
        fig,
        title="Mean generated code length vs model size",
        y_title="Mean response length (chars)",
        subplot_titles=[b.capitalize() for b in benchmarks],
    )


plot_mean_output_length(length_df)


## 6. Estimated mean latency (seconds per prompt)

All values are `EST_latency` placeholders (local Ollama + CodeGuard multi-agent overhead). Replace with measured timings from benchmark logs when available.

In [47]:
lat_df = latency_df()
lat_df

,benchmark,base_model,size_gb,codeguard,variant,mean_latency_s,source
0,instruct,gemma3:4b,3.3,False,EST_baseline,2.8,EST_latency
1,instruct,gemma3:4b,3.3,True,codeguard,11.0,EST_latency
2,instruct,gpt-oss:20b,14.0,False,EST_baseline,7.5,EST_latency
3,instruct,gpt-oss:20b,14.0,True,codeguard,26.0,EST_latency
4,instruct,gpt-oss:120b,65.0,False,EST_baseline,22.0,EST_latency
5,instruct,gpt-oss:120b,65.0,True,codeguard,68.0,EST_latency
6,autocomplete,gemma3:4b,3.3,False,EST_baseline,2.2,EST_latency
7,autocomplete,gemma3:4b,3.3,True,codeguard,9.5,EST_latency
8,autocomplete,gpt-oss:20b,14.0,False,EST_baseline,6.0,EST_latency
9,autocomplete,gpt-oss:20b,14.0,True,codeguard,22.0,EST_latency


In [48]:
def plot_estimated_latency(lat_df: pd.DataFrame) -> None:
    benchmarks = sorted(lat_df["benchmark"].unique())
    fig = make_subplots(
        rows=1,
        cols=len(benchmarks),
        subplot_titles=[f"{b.capitalize()} (EST)" for b in benchmarks],
    )

    for col, benchmark in enumerate(benchmarks, start=1):
        sub = lat_df[lat_df["benchmark"] == benchmark]
        for codeguard, label, color in [
            (False, "EST baseline", COLOR_BASELINE),
            (True, "EST CodeGuard", COLOR_CODEGUARD),
        ]:
            part = sub[sub["codeguard"] == codeguard].copy()
            part["variant"] = "EST_baseline"
            _add_series(
                fig,
                part,
                "size_gb",
                "mean_latency_s",
                name=label,
                color=color,
                row=1,
                col=col,
                codeguard=codeguard,
                est_prefix="EST<br>",
            )

    _finalize_subplots(
        fig,
        title="Estimated end-to-end latency vs model size",
        y_title="Mean latency (s, log scale)",
        log_y=False,
        subplot_titles=[f"{b.capitalize()} (EST)" for b in benchmarks],
    )


plot_estimated_latency(lat_df)
